# HCM AI Challenge: Colab setup

This notebook keeps mutable state on Google Drive. Run it once per fresh Colab runtime before indexing or search. It loads `.env` when present but never prints secrets; Colab Secrets and existing runtime variables take precedence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# These bootstrap paths can be supplied by the Colab runtime before this notebook starts.
DRIVE_ROOT = Path(os.environ.get('HCM_AI_DRIVE_ROOT') or '/content/drive/MyDrive/HCM-AI-Challenge-2026')
REPO_ROOT = Path(os.environ.get('HCM_AI_REPO_ROOT') or '/content/HCM-AI-Challenge-2026')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError(
        f'Clone or upload this repository to {REPO_ROOT}, then rerun this cell. '
        'Do not put the editable checkout inside a generated artifact folder.'
    )

In [ ]:
%cd {REPO_ROOT}
# Install runtime extras only in the ephemeral Colab VM. Model/download caches remain on Drive below.
!pip -q install -e ".[dev,retrieval,models,ocr,asr,gemini]"
# Git intentionally does not clone .env. If needed, run `!cp -n .env.example .env`,
# edit the new file in Colab's Files panel, then continue with the next cell.

In [ ]:
from hcm_ai.environment import load_environment

# Set HCM_AI_ENV_FILE before importing hcm_ai when the private file lives on Drive.
# Otherwise the repository's .env is loaded automatically when it exists.
LOADED_ENV = load_environment()
print('Environment file:', LOADED_ENV or 'not found; using runtime/default values')

def env_path(name, fallback):
    return Path(os.environ.get(name) or str(fallback)).expanduser()

DRIVE_ROOT = env_path('HCM_AI_DRIVE_ROOT', DRIVE_ROOT)
DATA_PATH = env_path('DATA_PATH', os.environ.get('AIC2025_ROOT') or DRIVE_ROOT / 'AIC2025')
DATA_ROOT = env_path('DATA_ROOT', DRIVE_ROOT / 'data')
ARTIFACT_ROOT = env_path('ARTIFACT_ROOT', DRIVE_ROOT / 'artifacts')
OUTPUT_ROOT = env_path('OUTPUT_ROOT', DRIVE_ROOT / 'outputs')
MODEL_CACHE = env_path('MODEL_CACHE', DRIVE_ROOT / 'model_cache')
for path in (DATA_ROOT, ARTIFACT_ROOT, OUTPUT_ROOT, MODEL_CACHE):
    path.mkdir(parents=True, exist_ok=True)

for name, value in {
    'DATA_PATH': DATA_PATH,
    'DATA_ROOT': DATA_ROOT,
    'ARTIFACT_ROOT': ARTIFACT_ROOT,
    'OUTPUT_ROOT': OUTPUT_ROOT,
    'MODEL_CACHE': MODEL_CACHE,
    'HF_HOME': MODEL_CACHE / 'huggingface',
    'TRANSFORMERS_CACHE': MODEL_CACHE / 'huggingface' / 'hub',
}.items():
    if not os.environ.get(name):
        os.environ[name] = str(value)
print({key: os.environ[key] for key in ('DATA_PATH', 'DATA_ROOT', 'ARTIFACT_ROOT', 'OUTPUT_ROOT', 'MODEL_CACHE')})

In [ ]:
# Optional: set GOOGLE_API_KEY from Colab Secrets. Never print it or save it to Drive.
try:
    from google.colab import userdata
    secret = userdata.get('GOOGLE_API_KEY')
    if secret:
        os.environ['GOOGLE_API_KEY'] = secret
except Exception:
    # Local/offline runs retain the deterministic heuristic fallback.
    pass

print('Gemini key available:', bool(os.environ.get('GOOGLE_API_KEY')))

In [ ]:
from hcm_ai.runtime import detect_capabilities, resolve_profile

PROFILE_REQUEST = os.environ.get('HCM_AI_PROFILE') or 'auto'
PROFILE = resolve_profile(PROFILE_REQUEST)
os.environ['HCM_AI_PROFILE'] = PROFILE
capabilities = detect_capabilities()
print('Resolved profile:', PROFILE)
print(capabilities)

# The profile can safely downgrade on a CPU runtime. Heavy models are loaded lazily by commands that use them.